# 07 Haze: does the 13-band advantage hold in hazy images?

**Purpose.** Answer SQ4: does the 13-band advantage still hold when the images are hazy, both with haze that affects every band equally and with haze that, as in reality, affects short wavelengths most? The saved models of notebooks 03 to 05 predict the same test patches with simulated haze of increasing thickness; nothing is retrained.

**Functions.** `src/piece_of_crop/haze.py` (`haze_share`, `add_haze`, `hazy_test_loader`, `rf_hazy_probabilities`, `WAVELENGTHS_NM`), `evaluate.py` (`insurer_cost_matrix`, `block_bootstrap_ci`), `train.py` (`predict`), `cnn.py` (`SmallCNN`) and `resnet.py` (`build_resnet`, `IMAGE_SIZE`).

**Reads.** `results/split.csv`, `results/band_stats.csv`, the clean test predictions in `results/predictions/`, the 18 saved models in `models/` and the patch cache `data/eurosat_cache.npy`.

**Saves.** `results/haze_runs.csv`, `results/haze_differences.csv` and the figures `results/figures/haze_*.png`. The predictions on hazy patches are not saved per patch, because the summary tables hold everything the report uses.

## 1. Setup

In [ ]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

from piece_of_crop import BAND_NAMES, CLASS_NAMES, MODEL_FILES, RGB_BAND_INDICES, read_split, get_cache
from piece_of_crop import SmallCNN, build_resnet, IMAGE_SIZE, predict
from piece_of_crop import WAVELENGTHS_NM, haze_share, add_haze, hazy_test_loader, rf_hazy_probabilities
from piece_of_crop import insurer_cost_matrix, block_bootstrap_ci

SEEDS = (0, 1, 2)
INPUTS = ("rgb", "13band")
MODELS = tuple(MODEL_FILES)  # RandomForest, SmallCNN, ResNet18
PROFILES = ("uniform", "wavelength")
AOD_LEVELS = (0.1, 0.3, 0.5, 0.8)  # aerosol optical depth at 550 nm, see section 2
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 0

project_root = Path.cwd().parents[0]
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"
models_dir = project_root / "models"

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")
test_rows = split[split["split"].eq("test")].reset_index(drop=True)
y_true = test_rows["label"].to_numpy()
blocks = test_rows["block"].to_numpy()
label_ids = np.arange(len(CLASS_NAMES))
costs = insurer_cost_matrix()
critical = costs >= 8  # as in notebook 06
airlight = band_stats["clip_high"].to_numpy(dtype=np.float32)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

print("Device:", device, "| test patches:", len(test_rows))

## 2. The haze model

**Claim.** We simulate haze with the standard model of a hazy image, with the haze per band set by physics and its thickness on a scale used to measure real haze, so that every step can be checked and every level means something.

**How it works.** A hazy value is the clear value seen through the haze plus the light that the haze scatters towards the sensor:

hazy = t × clear + (1 − t) × airlight

where t is the share of light that gets through. This is the model "widely used to describe the formation of a hazy image" (external: He, Sun and Tang, 2011, "Single Image Haze Removal Using Dark Channel Prior", *IEEE TPAMI* 33(12), equation 1). We build it from three parts:

1. **Thickness: aerosol optical depth (AOD) at 550 nm,** the standard measure of haze. We use four levels, one from each category of a NOAA guide: 0.1 (clean, 0 to 0.2), 0.3 (thin, 0.2 to 0.4), 0.5 (moderately thick, 0.4 to 0.6) and 0.8 (thick, 0.6 to 1.0) (external: NOAA, *VIIRS Aerosol Optical Depth Quick Guide*).
2. **Haze per band: the Ångström law.** The optical depth at wavelength λ is AOD × (λ / 550 nm)^−α. The exponent was introduced by Ångström in 1929, and its average for natural atmospheres is about 1.3, within a typical range of 0.5 to 2.5 (external: instesre.org/Aerosols/angstrom.htm). So B01 (443 nm) gets more haze than B03 (560 nm), and B12 (2,190 nm) far less. The central wavelengths of the bands are those published by the USGS (external: USGS EROS Archive, Sentinel-2).
3. **Transmission.** Light that crosses a layer of optical depth τ is reduced to exp(−τ) of its strength on a vertical path, and more on a slanted one (Beer–Lambert law; external: Penn State, METEO 300, section 6.10). The light crosses the haze twice, from the sun to the ground and back up to the satellite, so with the sun overhead t = exp(−2τ).

The **airlight** is the brightness of the haze itself. We set it to the brightest normal value of each band, the training 99.9th percentile from notebook 02, so that thick haze turns a patch bright and pale. The **uniform** profile gives every band the haze of 550 nm. It is not realistic; it is a control. If the 13 bands hold up better only under the wavelength profile, the reason is that the infrared bands see through the haze, not that there are more bands.

Haze is added to the raw bands, which are then clipped and normalised with the clean training statistics, exactly what a deployed model would receive. No model is retrained, as in the course's adversarial examples, where a trained model is given "slightly modified" inputs to see whether its predictions change (lecture_15_intro_to_generative_models.pdf, slide 8).

**Counterargument.** This is a simplification of real haze: the sun is rarely straight overhead in Europe, so the real path is longer than two vertical passes, real airlight is bluer than our per-band bright value, and EuroSAT images already contain the atmosphere of the day they were taken, so our levels are haze added on top. We accept these limits because each one can be stated and the table below shows exactly what is added to each band.

In [ ]:
# share of each band's value that comes from the haze, (1 - t)
share_table = pd.DataFrame({f"{profile}, AOD {aod}": haze_share(aod, profile) for profile in PROFILES for aod in AOD_LEVELS},
                           index=BAND_NAMES).T
order = [BAND_NAMES[k] for k in np.argsort(WAVELENGTHS_NM)]
print("Share of haze in each band (bands in wavelength order):")
display(share_table[order].round(2))

# one patch at every level, on a fixed display scale so the haze is visible
example = int(np.where(y_true == CLASS_NAMES.index("PermanentCrop"))[0][0])
clear = np.asarray(cache[test_rows["cache_row"][example]], dtype=np.float32)
fig, axes = plt.subplots(2, len(AOD_LEVELS) + 1, figsize=(13, 5.6))
for row, profile in enumerate(PROFILES):
    for col, aod in enumerate((0.0,) + AOD_LEVELS):
        image = add_haze(clear, aod, profile, airlight)[RGB_BAND_INDICES].transpose(1, 2, 0)
        axes[row, col].imshow(np.clip(image / 3000, 0, 1))
        axes[row, col].set_title(f"{profile}, AOD {aod}", fontsize=9)
        axes[row, col].axis("off")
fig.suptitle("One PermanentCrop test patch in true colour (bands 4, 3, 2) with increasing haze")
fig.tight_layout()
fig.savefig(figures_dir / "haze_example_patch.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Predict every saved model on the hazy test patches

For each of the 18 saved models (3 models x 2 inputs x 3 seeds), the clean test set is predicted first and compared with the predictions saved in notebooks 03 to 05: they must agree, otherwise the haze results would not be comparable with notebook 06. Then each model predicts the test set at the four haze levels of both profiles. For every run we keep accuracy, macro-F1, the critical-error rate and cost per patch of notebook 06, and how often HerbaceousVegetation is called Forest, the confusion that notebook 06 linked to the extra bands.

In [ ]:
def load_model(model, input_type, seed):
    prefix = MODEL_FILES[model]
    if model == "RandomForest":
        forest = joblib.load(models_dir / f"{prefix}_{input_type}_seed{seed}.joblib")
        assert list(forest.classes_) == list(label_ids)  # probability columns are classes 0 to 9
        return forest
    channels = 3 if input_type == "rgb" else 13
    net = SmallCNN(in_channels=channels, num_classes=len(CLASS_NAMES)) if model == "SmallCNN" else \
        build_resnet(in_channels=channels, num_classes=len(CLASS_NAMES), pretrained=False)
    net.load_state_dict(torch.load(models_dir / f"{prefix}_{input_type}_seed{seed}.pt", map_location="cpu"))
    return net.to(device)


def predict_hazy(model_name, model, input_type, aod, profile):
    if model_name == "RandomForest":
        return rf_hazy_probabilities(model, test_rows, cache, band_stats, input_type, aod, profile, airlight).argmax(axis=1)
    image_size = IMAGE_SIZE if model_name == "ResNet18" else None
    loader = hazy_test_loader(test_rows, cache, band_stats, input_type, aod, profile, airlight, image_size)
    labels, proba = predict(model, loader, device)
    assert np.array_equal(labels, y_true)  # same patches in the same order
    return proba.argmax(axis=1)


herb, forest_id = CLASS_NAMES.index("HerbaceousVegetation"), CLASS_NAMES.index("Forest")
hazy_predictions, run_rows = {}, []
for model_name in MODELS:
    for input_type in INPUTS:
        for seed in SEEDS:
            start = time.time()
            model = load_model(model_name, input_type, seed)
            saved = pd.read_csv(predictions_dir / f"{MODEL_FILES[model_name]}_{input_type}_seed{seed}.csv")["y_pred"].to_numpy()
            clean = predict_hazy(model_name, model, input_type, 0.0, "uniform")
            agreement = (clean == saved).mean()
            assert agreement >= 0.999, f"clean predictions differ from the saved ones ({agreement:.4f})"
            conditions = [("none", 0.0, clean)] + [(profile, aod, predict_hazy(model_name, model, input_type, aod, profile))
                                                   for profile in PROFILES for aod in AOD_LEVELS]
            for profile, aod, y_pred in conditions:
                hazy_predictions[(model_name, input_type, seed, profile, aod)] = y_pred
                run_rows.append({"model": model_name, "input": input_type, "seed": seed, "profile": profile, "aod": aod,
                                 "accuracy": accuracy_score(y_true, y_pred),
                                 "macro_f1": f1_score(y_true, y_pred, average="macro"),
                                 "critical_error_rate": critical[y_true, y_pred].mean(),
                                 "cost_per_patch": costs[y_true, y_pred].mean(),
                                 "herb_called_forest": int(((y_true == herb) & (y_pred == forest_id)).sum())})
            print(f"{model_name} {input_type} seed {seed}: clean agreement {agreement:.4f}, {time.time() - start:.0f} s")

haze_runs = pd.DataFrame(run_rows)
haze_runs.to_csv(results_dir / "haze_runs.csv", index=False)

## 4. Accuracy and critical errors as the haze thickens

Mean over the three seeds. AOD 0 is the clean test set; solid lines are 13 bands, dashed lines RGB.

In [ ]:
# clean results count for both profiles
clean_rows = haze_runs[haze_runs["profile"].eq("none")]
curves = pd.concat([haze_runs[~haze_runs["profile"].eq("none")]] + [clean_rows.assign(profile=p) for p in PROFILES])
means = curves.groupby(["model", "input", "profile", "aod"])[["accuracy", "critical_error_rate"]].mean().reset_index()

fig, axes = plt.subplots(2, len(MODELS), figsize=(15, 8), sharex=True)
colours = {"uniform": "tab:grey", "wavelength": "tab:blue"}
for col, model_name in enumerate(MODELS):
    for row, (measure, label) in enumerate((("accuracy", "Accuracy"), ("critical_error_rate", "Critical-error rate"))):
        ax = axes[row, col]
        for profile in PROFILES:
            for input_type in INPUTS:
                line = means[(means["model"] == model_name) & (means["profile"] == profile) & (means["input"] == input_type)]
                ax.plot(line["aod"], line[measure], marker="o", color=colours[profile],
                        ls="-" if input_type == "13band" else "--", label=f"{input_type}, {profile} haze")
        ax.set(title=f"{model_name}: {label.lower()}", ylabel=label)
        ax.grid(alpha=0.3)
        if row == 1:
            ax.set_xlabel("Added haze (aerosol optical depth at 550 nm)")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Test results with simulated haze (mean of 3 seeds; solid = 13 bands, dashed = RGB)")
fig.tight_layout()
fig.savefig(figures_dir / "haze_curves.png", dpi=150, bbox_inches="tight")
plt.show()

display(means.pivot_table(index=["model", "profile", "aod"], columns="input", values="accuracy", sort=False)[list(INPUTS)].round(3))

## 5. The 13-band advantage at each haze level, with intervals

As in notebook 06: for every model and condition, the 13-band minus RGB difference per test patch, averaged over the seeds, with a 95% block-bootstrap interval. A positive accuracy difference and a negative critical-error difference favour the 13 bands. If the advantage holds under haze, the intervals of the random forest and the CNN stay away from 0 as the haze thickens.

In [ ]:
rng = np.random.default_rng(BOOTSTRAP_SEED)
conditions = [("none", 0.0)] + [(p, a) for p in PROFILES for a in AOD_LEVELS]
difference_rows = []
for model_name in MODELS:
    for profile, aod in conditions:
        for measure, per_patch in (("accuracy", lambda p: (p == y_true).astype(float)),
                                   ("critical_error_rate", lambda p: critical[y_true, p].astype(float))):
            per_seed = np.array([per_patch(hazy_predictions[(model_name, "13band", s, profile, aod)])
                                 - per_patch(hazy_predictions[(model_name, "rgb", s, profile, aod)]) for s in SEEDS])
            low, high = block_bootstrap_ci(per_seed.mean(axis=0), blocks, N_BOOTSTRAP, rng)
            difference_rows.append({"model": model_name, "profile": profile, "aod": aod, "measure": measure,
                                    "13band_minus_rgb": per_seed.mean(), "ci_low": low, "ci_high": high,
                                    "excludes_0": not (low <= 0 <= high)})

haze_differences = pd.DataFrame(difference_rows)
haze_differences.to_csv(results_dir / "haze_differences.csv", index=False)
display(haze_differences[haze_differences["measure"].eq("accuracy")].round(4))
display(haze_differences[haze_differences["measure"].eq("critical_error_rate")].round(4))

## 6. Does haze make grassland look like forest?

Notebook 06 found that the HerbaceousVegetation patches the 13-band ResNet called Forest were brighter in B01, the band most sensitive to haze. If haze pushes grassland towards forest, this confusion should grow with the haze. The table shows it per seed for every model and input (the test set has 428 HerbaceousVegetation patches).

In [ ]:
herb_forest = curves.groupby(["model", "input", "profile", "aod"])["herb_called_forest"].mean()
display(herb_forest.unstack(["input"]).unstack(["profile"]).round(1))